# Support Tickets - Silver Pipeline

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import reduce

print("[INFO] Support Tickets Silver pipeline started")

try:
    dbutils.widgets.text("batch_id", "")
    batch_id = dbutils.widgets.get("batch_id")
except:
    batch_id = None

print("[INFO] Databricks Spark session ready")
spark
print(f"[INFO] Batch ID: {batch_id}")

## 1. Read Bronze support_tickets

In [0]:
bronze_path = "/Volumes/datalake_catalog/datalake_schema/bronze/support_tickets"

df_bronze = (
    spark.read.format("delta").load(bronze_path)
    .drop("dw_ingested_at", "dw_source_file", "dw_batch_id", "batch_id", "source_table")
)

df_bronze.show(5, truncate=False)
df_bronze.printSchema()

## 2. Cast & Normalize

In [0]:
df_casted = df_bronze.select(
    F.expr("try_cast(ticket_id as bigint)").alias("ticket_id"),
    F.expr("try_cast(user_id as bigint)").alias("user_id"),
    F.col("category").cast("string"),
    F.col("description").cast("string"),
    F.expr("try_cast(created_at as timestamp)").alias("created_at"),
    F.expr("try_cast(ingest_time as timestamp)").alias("ingest_time")
)

df_validated = df_casted.withColumn(
    "validation_error",
    F.concat_ws(
        "; ",
        F.when(F.col("ticket_id").isNull(), "invalid ticket_id"),
        F.when(F.col("user_id").isNull(), "invalid user_id"),
        F.when(F.col("created_at").isNull(), "invalid created_at"),
        F.when(F.col("category").isNull(), "invalid category")
    )
)

df_valid = df_validated.filter(F.col("validation_error") == "")
df_quarantine = df_validated.filter(F.col("validation_error") != "")

print(df_valid.count(), df_quarantine.count())
df1 = df_valid

## 3. Validation

In [0]:
valid_ticket_categories = ["billing", "technical", "cancellation", "feature_request"]

df_validated = df1.withColumn(
    "validation_error",
    F.concat_ws(
        "; ",
        F.when(F.col("ticket_id").isNull(), "ticket_id is null"),
        F.when(F.col("user_id").isNull(), "user_id is null"),
        F.when(F.col("created_at").isNull(), "created_at is null"),
        F.when(~F.col("category").isin(valid_ticket_categories), "invalid category")
    )
)

df_valid = df_validated.filter(F.col("validation_error") == "")
df_quarantine_1 = df_validated.filter(F.col("validation_error") != "")

print(df_valid.count(), df_quarantine_1.count())

## 4. Deduplication

In [0]:
w = Window.partitionBy("ticket_id").orderBy(F.col("ingest_time").desc())

df_ranked = df_valid.withColumn("rn", F.row_number().over(w))

df2 = df_ranked.filter(F.col("rn") == 1).drop("rn", "validation_error")

df_quarantine_2 = (
    df_ranked.filter(F.col("rn") > 1)
    .withColumn("validation_error", F.lit("duplicate ticket_id"))
    .drop("rn")
)

print(df2.count(), df_quarantine_2.count())

## 5. Validate user_id

In [0]:
silver_users_path = "/Volumes/datalake_catalog/datalake_schema/silver/users"

df_users = (
    spark.read.format("delta").load(silver_users_path)
    .select(F.col("user_id").cast("bigint").alias("user_id"))
    .dropDuplicates()
)

df3 = df2.join(df_users, on="user_id", how="inner")

df_quarantine_3 = (
    df2.join(df_users, on="user_id", how="left_anti")
    .withColumn("validation_error", F.lit("user_id not found"))
)

print(df3.count(), df_quarantine_3.count())

## 6. Combine quarantine

In [0]:
df_quarantine_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    [df_quarantine_1, df_quarantine_2, df_quarantine_3]
)

print(df_quarantine_all.count())

## 7. Silver dataset

In [0]:
df_silver = df3

## 8. Upsert to Silver Delta

In [0]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/support_tickets"

df_upsert = df_silver

w = Window.partitionBy("ticket_id").orderBy(F.col("created_at").desc(), F.col("ingest_time").desc())

df_upsert = (
    df_upsert.withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

if DeltaTable.isDeltaTable(spark, silver_path):
    print("[INFO] Merge into existing silver table")
    delta = DeltaTable.forPath(spark, silver_path)

    delta.alias("t").merge(
        df_upsert.alias("s"),
        "t.ticket_id = s.ticket_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

else:
    print("[INFO] Creating silver table")
    df_upsert.write.format("delta").mode("overwrite").save(silver_path)

## 9. Done

In [0]:
print("[DONE] support_tickets silver pipeline completed")